# Análisis exploratorio Wikidata

In [2]:
from pyspark.sql import SparkSession, functions as F, types as T

spark = (
    SparkSession.builder
    .appName("eda_wikidata_festivals")
    .enableHiveSupport()
    .getOrCreate()
)

RAW_PATH = "/Obligatorio/landing/wikidata/festivals.csv"  # ajustar nombre si corresponde

# Todos los campos llegan como string: la fuente es un CSV de una consulta SPARQL.
festivals_schema = T.StructType([
    T.StructField("festival",                            T.StringType(), True),  # URI Wikidata
    T.StructField("festivalLabel",                       T.StringType(), True),
    T.StructField("countryLabel",                        T.StringType(), True),
    T.StructField("coord",                               T.StringType(), True),  # Point(lon lat)
    T.StructField("genreLabel",                          T.StringType(), True),
    T.StructField("setlist_fm_festival_ID",              T.StringType(), True),
    T.StructField("identificador_MusicBrainz_de_serie",  T.StringType(), True),
])

festivals_raw = (
    spark.read
    .option("header", True)
    .option("encoding", "UTF-8")
    .schema(festivals_schema)
    .csv(RAW_PATH)
)



In [3]:
print("Filas:", festivals_raw.count())
print("Columnas:", len(festivals_raw.columns))
print("Nombres de columnas:", festivals_raw.columns)

festivals_raw.printSchema()
festivals_raw.show(10, truncate=False)


Filas: 1112
Columnas: 7
Nombres de columnas: ['festival', 'festivalLabel', 'countryLabel', 'coord', 'genreLabel', 'setlist_fm_festival_ID', 'identificador_MusicBrainz_de_serie']
root
 |-- festival: string (nullable = true)
 |-- festivalLabel: string (nullable = true)
 |-- countryLabel: string (nullable = true)
 |-- coord: string (nullable = true)
 |-- genreLabel: string (nullable = true)
 |-- setlist_fm_festival_ID: string (nullable = true)
 |-- identificador_MusicBrainz_de_serie: string (nullable = true)

+--------------------------------------+--------------------+------------+--------------------------------+-----------------------+----------------------+------------------------------------+
|festival                              |festivalLabel       |countryLabel|coord                           |genreLabel             |setlist_fm_festival_ID|identificador_MusicBrainz_de_serie  |
+--------------------------------------+--------------------+------------+------------------------------

In [4]:
# Los faltantes vienen como NULL o cadena vacía. Esperamos MUCHOS nulos en
# genreLabel y en los identificadores externos: en la consulta SPARQL son OPTIONAL,
# así que su ausencia es legítima y no representa un error de calidad.
MARCADORES = ["", "\\N"]

def reporte_nulos(df, titulo):
    total = df.count()
    filas = []
    for c in df.columns:
        col_str = F.trim(F.col(c).cast("string"))
        nulos = df.filter(F.col(c).isNull() | col_str.isin(MARCADORES)).count()
        pct = round(100 * nulos / total, 1) if total else 0.0
        filas.append((c, nulos, pct))
    print("\n" + titulo + f"  (total filas: {total})")
    (spark.createDataFrame(filas, ["columna", "nulos", "porcentaje"])
          .orderBy(F.desc("nulos"))
          .show(100, truncate=False))

reporte_nulos(festivals_raw, "NULOS EN festivals_raw")



NULOS EN festivals_raw  (total filas: 1112)


[Stage 28:>                                                         (0 + 2) / 2]

+----------------------------------+-----+----------+
|columna                           |nulos|porcentaje|
+----------------------------------+-----+----------+
|setlist_fm_festival_ID            |1086 |97.7      |
|identificador_MusicBrainz_de_serie|850  |76.4      |
|genreLabel                        |790  |71.0      |
|countryLabel                      |75   |6.7       |
|coord                             |0    |0.0       |
|festival                          |0    |0.0       |
|festivalLabel                     |0    |0.0       |
+----------------------------------+-----+----------+



In [5]:
# Extraemos el identificador del festival desde la URI.
fest = festivals_raw.withColumn(
    "wikidata_id", F.regexp_extract("festival", r"/entity/(Q[0-9]+)", 1)
)

total = fest.count()
distintos = fest.select("wikidata_id").distinct().count()
print("Filas totales:", total)
print("Festivales distintos (wikidata_id):", distintos)
print("Filas exactas duplicadas:", total - fest.dropDuplicates().count())

# Festivales que aparecen en más de una fila
print("\nFestivales con más de una fila:")
(fest.groupBy("wikidata_id").count()
     .filter(F.col("count") > 1)
     .orderBy(F.desc("count"))
     .show(10, truncate=False))

# ¿Por qué se repiten? Por múltiples géneros y/o múltiples coordenadas.
multi_genero = (fest.groupBy("wikidata_id")
    .agg(F.countDistinct("genreLabel").alias("g"))
    .filter(F.col("g") > 1).count())
multi_coord = (fest.groupBy("wikidata_id")
    .agg(F.countDistinct("coord").alias("c"))
    .filter(F.col("c") > 1).count())
print("Festivales con >1 género:", multi_genero)
print("Festivales con >1 coordenada:", multi_coord)



Filas totales: 1112
Festivales distintos (wikidata_id): 992
Filas exactas duplicadas: 0

Festivales con más de una fila:
+-----------+-----+
|wikidata_id|count|
+-----------+-----+
|Q914820    |12   |
|Q5596999   |9    |
|Q836514    |9    |
|Q108882045 |7    |
|Q3417886   |6    |
|Q109289746 |6    |
|Q28040033  |5    |
|Q2738110   |5    |
|Q582456    |4    |
|Q3070023   |4    |
+-----------+-----+
only showing top 10 rows

Festivales con >1 género: 39
Festivales con >1 coordenada: 3


In [6]:
# Formato del identificador Wikidata
fest = fest.withColumn("valid_wikidata_id", F.col("wikidata_id").rlike("^Q[0-9]+$"))
print("wikidata_id con formato inválido:",
      fest.filter(~F.col("valid_wikidata_id")).count())

# Parseo de coord: Point(longitud latitud) -> dos columnas numéricas
fest = (fest
    .withColumn("longitude", F.regexp_extract("coord", r"Point\((-?[0-9.]+) (-?[0-9.]+)\)", 1).cast("double"))
    .withColumn("latitude",  F.regexp_extract("coord", r"Point\((-?[0-9.]+) (-?[0-9.]+)\)", 2).cast("double")))

print("coord que NO se pudieron parsear:",
      fest.filter(F.col("longitude").isNull() | F.col("latitude").isNull()).count())
print("Coordenadas fuera de rango geográfico:",
      fest.filter(~(F.col("latitude").between(-90, 90) &
                    F.col("longitude").between(-180, 180))).count())


wikidata_id con formato inválido: 0
coord que NO se pudieron parsear: 0
Coordenadas fuera de rango geográfico: 0


In [7]:
# Cantidad de países y géneros distintos (contexto del dominio)
print("Países distintos:",
      festivals_raw.select("countryLabel")
                   .filter(F.col("countryLabel").isNotNull() & (F.trim(F.col("countryLabel")) != ""))
                   .distinct().count())
print("Géneros distintos:",
      festivals_raw.select("genreLabel")
                   .filter(F.col("genreLabel").isNotNull() & (F.trim(F.col("genreLabel")) != ""))
                   .distinct().count())

# Posible mojibake (Ã / Â) en textos con tildes -> justifica corrección de codificación
moji = festivals_raw.filter(
    F.col("festivalLabel").rlike("Ã|Â") | F.col("countryLabel").rlike("Ã|Â") | F.col("genreLabel").rlike("Ã|Â")
).count()
print("Textos con posible mojibake (Ã/Â):", moji)

# Ejemplo concreto de festival con varias filas (país vacío + 2 coordenadas + 2 géneros)
print("\nEjemplo San Remo (Q206959):")
(festivals_raw
    .withColumn("wikidata_id", F.regexp_extract("festival", r"/entity/(Q[0-9]+)", 1))
    .filter(F.col("wikidata_id") == "Q206959")
    .select("festivalLabel", "countryLabel", "coord", "genreLabel")
    .show(truncate=False))


Países distintos: 64
Géneros distintos: 102
Textos con posible mojibake (Ã/Â): 0

Ejemplo San Remo (Q206959):
+----------------------------------+------------+-------------------------------+------------------+
|festivalLabel                     |countryLabel|coord                          |genreLabel        |
+----------------------------------+------------+-------------------------------+------------------+
|Festival de la Canción de San Remo|null        |Point(7.775 43.8175)           |show de televisión|
|Festival de la Canción de San Remo|null        |Point(7.777546924 43.817662414)|show de televisión|
|Festival de la Canción de San Remo|null        |Point(7.775 43.8175)           |televisión musical|
|Festival de la Canción de San Remo|null        |Point(7.777546924 43.817662414)|televisión musical|
+----------------------------------+------------+-------------------------------+------------------+



In [8]:
# 'is_us_festival' identifica los festivales de EE. UU. para la pregunta de impacto aéreo.
es_us = festivals_raw.filter(
    F.lower(F.col("countryLabel")).isin(
        "estados unidos", "estados unidos de américa",
        "united states", "united states of america"
    )
)
print("Festivales en Estados Unidos:", es_us.count())
es_us.select("festivalLabel", "countryLabel", "coord").show(10, truncate=False)


Festivales en Estados Unidos: 177
+---------------------------------------+--------------+----------------------------------+
|festivalLabel                          |countryLabel  |coord                             |
+---------------------------------------+--------------+----------------------------------+
|Altamont Speedway Free Festival        |Estados Unidos|Point(-121.561111111 37.739444444)|
|Ann Arbor Blues and Jazz Festival      |Estados Unidos|Point(-83.79472222 42.21138889)   |
|Mantra-Rock Dance                      |Estados Unidos|Point(-122.421 37.7876)           |
|US Festival                            |Estados Unidos|Point(-117.402 34.204)            |
|Music Academy of the West              |Estados Unidos|Point(-119.648888888 34.419444444)|
|80/35 Music Festival                   |Estados Unidos|Point(-93.6346 41.5853)           |
|Agape Music Festival                   |Estados Unidos|Point(-89.4082 38.8927)           |
|Harrisburg Independence Day Celebration|Estad